# OpenMarket Quickstart

Walks through loading the [sample split](https://huggingface.co/datasets/gregyoung14/openmarket-btc-polymarket) of the OpenMarket BTC Polymarket dataset, inspecting each table, and joining Binance trades with Polymarket ticks to compute a 1-minute mid-price series.

Run from the repo root:

```bash
.venv/bin/jupyter nbconvert --to notebook --execute notebooks/quickstart.ipynb
```

In [ ]:
from pathlib import Path
from huggingface_hub import snapshot_download
import pyarrow.parquet as pq
import pandas as pd

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 140)

REPO = "gregyoung14/openmarket-btc-polymarket"

In [ ]:
root = Path(snapshot_download(
    REPO, repo_type="dataset",
    allow_patterns=["sample/**", "metadata/**", "README.md"],
))
print(f"downloaded -> {root}")
print(f"parquet files: {len(list(root.rglob('*.parquet')))}")

In [ ]:
tables = {}
for p in sorted(root.rglob("*.parquet")):
    rel = p.relative_to(root)
    name = rel.parts[1] if rel.parts[0] == "sample" else rel.parts[0]
    tables.setdefault(name, []).append(p)

for name, files in sorted(tables.items()):
    df = pd.concat([pq.read_table(p).to_pandas() for p in files], ignore_index=True)
    print(f"{name:<25}  rows={len(df):>6}  cols={len(df.columns):>2}  bytes={sum(p.stat().st_size for p in files):>8,}")

In [ ]:
trades = pd.concat(
    [pq.read_table(p).to_pandas() for p in tables["binance_trades"]], ignore_index=True
).sort_values("trade_time")
print(trades.head())
print(f"\ntime range: {trades['trade_time'].min()}  ->  {trades['trade_time'].max()}")

In [ ]:
poly = pd.concat(
    [pq.read_table(p).to_pandas() for p in tables["polymarket_ticks_ms"]], ignore_index=True
)
print(poly.columns.tolist())
print(poly.head(3))

if {"best_bid", "best_ask"}.issubset(poly.columns):
    mid = (poly["best_bid"] + poly["best_ask"]) / 2
    poly["mid"] = mid
    poly_minute = (poly.set_index(pd.to_datetime(poly["source_ts_ms"], unit="ms"))["mid"].resample("1min").mean())
    print(f"\n1-minute mid-price points: {poly_minute.notna().sum()}")
    print(poly_minute.tail())

In [ ]:
if "lag_pairs_ms" in tables:
    lag = pd.concat(
        [pq.read_table(p).to_pandas() for p in tables["lag_pairs_ms"]], ignore_index=True
    )
    print(lag.head())
    if "lead_lag_ms" in lag.columns:
        print(f"\nlead_lag_ms stats (positive = polymarket follows binance):\n{lag['lead_lag_ms'].describe()}")

## Next steps

- Load the `full/` split once it's published for larger backtests
- Try the `backtester` crate on this sample data
- See `paper/paper.md` for the systems paper describing the sync pipeline
- See `research/legacy-ml/` for example signal strategies